In [0]:
from pyspark.sql.functions import col,upper,to_date,when,coalesce,lit,current_timestamp,try_to_date
# 1. Cargar datos desde la base de datos que creamos en SQL
df_sale_bronze = spark.table("retail_peru_db.sales_bronze")

# 2. Limpieza de la tabla de Ventas
df_sales_silver = df_sale_bronze\
    .filter(col("monto_total").isNotNull())\
    .withColumn("distrito_limpio", upper(col("distrito")))\
    .withColumn("fecha_venta_dt",
        coalesce(
            try_to_date(col("fecha_venta"), "yyyy-MM-dd"),
            try_to_date(col("fecha_venta"), "dd/MM/yyyy")
            )
    )\
    .select(
        col("id_transaccion"),
        col("fecha_venta_dt").alias("fecha_venta"),
        col("tienda_nombre"),
        col("monto_total"),
        col("metodo_pago"),
        current_timestamp().alias("ingestion_timestamp")
    )

df_sales_silver = df_sales_silver.dropDuplicates(["id_transaccion"])

df_sales_silver.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("retail_peru_db.sales_silver")

display(spark.table("retail_peru_db.sales_silver"))

# CAPA SILVER (CUARENTENA DE DATOS ERRONEOS)

In [0]:
from pyspark.sql.functions import col, upper, to_date, coalesce, current_timestamp, lit, try_to_date, when
df_sale_raw = spark.table("retail_peru_db.sales_bronze")

# 2. Identificar Registros de Cuarentena (Data Errónea)
df_cuarentena = df_sale_raw.filter(
    col("monto_total").isNull() |
    (try_to_date(col("fecha_venta"), "yyyy-MM-dd").isNull() & try_to_date(col("fecha_venta"), "dd/MM/yyyy").isNull())
).withColumn("error_reason",
             when(col("monto_total").isNull(),lit("Monto"))
             .otherwise(lit("Formato de Fecha Inválido"))
)
# 3. Identificar Registros Válidos (Data Limpia)
df_sale_valido = df_sale_raw.filter(
    col("monto_total").isNotNull() &
    (try_to_date(col("fecha_venta"), "yyyy-MM-dd").isNotNull() | try_to_date(col("fecha_venta"), "dd/MM/yyyy").isNotNull())
)

# 4. Transformación de los Datos Válidos
df_sales_silver = df_sale_valido\
    .withColumn("distrito",upper(col("distrito")))\
    .withColumn("fecha", coalesce(
        try_to_date(col("fecha_venta"), "yyyy-MM-dd"),
        try_to_date(col("fecha_venta"), "dd/MM/yyyy")
    ))\
    .select(
        "id_transaccion", "fecha", "tienda_nombre", 
        "distrito", "monto_total", "metodo_pago"
    ).dropDuplicates(["id_transaccion"])

# 5. Guardar ambas tablas en el catálogo
df_sales_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("retail_peru_db.sales_silver")
df_cuarentena.write.format("delta").mode("overwrite").saveAsTable("retail_peru_db.sales_cuarentena")

print(f"✅ Procesamiento Silver terminado.")
print(f"📈 Registros limpios: {df_sales_silver.count()}")
print(f"⚠️ Registros en cuarentena: {df_cuarentena.count()}")